🏢 Proyecto: Agencia de Consultoría IA Autónoma
(Multi-Agente Jerárquico)
El objetivo: Crear un sistema multi-agente que funcione como un equipo especializado gestionado por un Agente Supervisor. El sistema será capaz de procesar solicitudes complejas como:

"Investiga el mercado de vehículos eléctricos en México, analiza los precios, genera una gráfica de barras y entrégame un reporte en PDF".

¿Por qué es un gran reto y qué aprenderás?
🧠 Enrutamiento por Supervisor
En lugar de un flujo lineal simple, aprenderás a construir un nodo Supervisor impulsado por LLM. Este nodo analiza la entrada del usuario y decide dinámicamente a qué sub-agente delegar la tarea:

Investigador: Búsqueda y extracción de datos.

Analista de Datos: Procesamiento y generación de visualizaciones.

Redactor: Consolidación de la información en formatos finales.

✋ Human-in-the-Loop (HIL) Nativo
Implementaremos las interrupciones nativas de LangGraph. El grafo pausará su ejecución automáticamente antes de realizar acciones críticas (como ejecutar código Python o enviar correos), solicitando tu aprobación explícita desde la interfaz antes de continuar.

### Configurar Cliente

In [8]:
from agents import AsyncOpenAI, OpenAIChatCompletionsModel, Agent
from dotenv import load_dotenv
import os

In [3]:
load_dotenv(override=True)

groq_api_key = os.getenv("GROQ_API_KEY")

google_api_key = os.getenv("GOOGLE_API_KEY")

hf_api_key = os.getenv("HF_API_KEY")

if groq_api_key:
    print(f'GROQ_API_KEY: {groq_api_key[:8]}')
else:
    print('GROQ_API_KEY not set')
if google_api_key:
    print(f'GOOGLE_API_KEY: {google_api_key[:8]}')
else:
    print('GOOGLE_API_KEY not set')
if hf_api_key:
    print(f'HF_API_KEY: {hf_api_key[:8]}')
else:
    print('HF_API_KEY not set')

GROQ_API_KEY: gsk_oFCs
GOOGLE_API_KEY: AIzaSyCl
HF_API_KEY: hf_ZPuYC


In [4]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
HF_BASE_URL = "https://api-inference.huggingface.co/models/deepseek-ai/DeepSeek-V2"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
LOCAL_URL = "http://localhost:11434/v1"

In [6]:
deepseek_client = AsyncOpenAI(base_url=HF_BASE_URL, api_key=hf_api_key)
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)
local_client = AsyncOpenAI(base_url=LOCAL_URL, api_key="ollama")

In [7]:
deepseek_model = OpenAIChatCompletionsModel(model="deepseek-ai/DeepSeek-V2", openai_client=deepseek_client)
gemini_model = OpenAIChatCompletionsModel(model="gemini-2.0-flash", openai_client=gemini_client)
groq_model = OpenAIChatCompletionsModel(model="llama-3-8b", openai_client=groq_client)
local_model = OpenAIChatCompletionsModel(model="llama3.1:8b", openai_client=local_client)

### Agente Investigador

In [ ]:
from langchain_community.utilities import GoogleSerperAPIWrapper
from agents import function_tool
from pydantic import BaseModel, Field
from typing import Dict

In [ ]:
# Formato de Busqueda y extraccion de datos
class DeepResearch(BaseModel):
    datos: Dict[str, float] = Field(description="Diccionario con etiquetas y valores numéricos representativos de los datos investigados.")
    contexto: str = Field(description="Descripción breve del contexto de los datos investigados (unidades, fuente, período)")
    sugerencia_visualizacion: str = Field(description="Sugerencia sobre qué aspecto sería más interesante mostrar en una gráfica")

In [ ]:
@function_tool
def tool_searcher(query : str ) -> str:
    """Busca información en la web, investiga sobre un tema en específico (Investigación profunda)"""
    try:
        serper = GoogleSerperAPIWrapper()
        result = serper.run(query)
        if not result:
            return "⚠️'tool_searcher' = No results found"
        return result
    except Exception as ex:
        return f"⚠️ Unexpected Exception in tool 'tool_searcher' = {str(ex)}"

In [ ]:
prompt_searcher = """
    ### Eres un agente investigador experto en recopilar y estructurar datos, dónde el usuario te puede hacer preguntas sobre un tema en concreto
        y tu deber es usar la herramienta 'tool_searcher' para investigar sobre lo que pida el usuario. \

    ### Cuando recibas un tema:
    1. SIEMPRE usa la herramienta 'tool_searcher' para investigar datos numéricos representativos y actuales sobre ese tema. \
    2. Estructura los datos como un diccionario Python {etiqueta: valor_numérico}. \
    3. Incluye entre 4 y 12 puntos de datos relevantes. \
    4. Describe brevemente el contexto de los datos (unidades, fuente, período). \
    5. ANTES de usar la herramienta 'tool_searcher' si te hace falta más contexto para la búsqueda siempre pregunta primero. \
    6. NUNCA investigues por tu cuenta, siempre usa la herramienta 'tool_searcher' para usar siempre los datos mas actualizados. \
    7. Responde basándote en lo que encontraste. \
    8. Sé cálido, empático y usa emojis apropiados. \
    9. Es OBLIGATORIO que preguntes por más contexto antes de usar la herramienta 'tool_searcher'.

     ### Criterios para visualización:
    - 'bar'  → comparaciones entre categorías, rankings
    - 'line' → evolución temporal, tendencias
    - 'pie'  → composición porcentual (máximo 7 categorías)
    - 'scatter' → relación entre variables
    - 'hist' → distribución de una variable continua
"""

In [ ]:
# modelo local -> investigador
agente_investigador = Agent(
    name="Agente Investigador",
    instructions=prompt_searcher,
    output_type=DeepResearch,
    tools=[tool_searcher],
    model=local_model,
    handoff_description="Agente de investigación profunda en la web que investiga y estructura datos sobre un tema en espeífico"
)

### Agente Analista de Datos

In [ ]:
from matplotlib import pyplot as plt
import matplotlib.ticker as mticker
import io, base64

In [ ]:
@function_tool
def plot_data(data: dict, chart_type: str, title: str, x_label: str = "", y_label: str = "") -> str:
    """ Genera una gráfica a partir de los datos recaudados.
    Parámetros
    ----------
    data       : dict[str, float]  — claves = etiquetas, valores = números.
    chart_type : Tipo de gráfica. Elige el más adecuado según el contexto:
                 - 'bar'     → comparar categorías (rankings, totales por grupo)
                 - 'line'    → tendencia temporal o evolución continua
                 - 'pie'     → proporciones de un todo (máximo 7 categorías)
                 - 'scatter' → correlación entre valores
                 - 'hist'    → distribución de frecuencias
                 Nunca uses 'pie' con más de 7 valores.
    title      : Título descriptivo de la gráfica.
    x_label    : Etiqueta del eje X (vacío para pie/hist).
    y_label    : Etiqueta del eje Y (vacío para pie).
    """
    try:
        labels = list(data.keys())
        values = [float(v) for v in data.values()]

        if not values:
            return "Sin datos para graficar."

        COLORS = ["#f0b429", "#26a69a", "#ef5350", "#42a5f5",
                  "#ab47bc", "#ff7043", "#66bb6a", "#26c6da"]
        MUTED  = "#5d6378"

        plt.style.use("dark_background")
        fig, ax = plt.subplots(figsize=(10, 5))
        fig.patch.set_facecolor("#131722")
        ax.set_facecolor("#131722")

        if chart_type == "bar":
            colors = [COLORS[i % len(COLORS)] for i in range(len(values))]
            bars = ax.bar(labels, values, color=colors, edgecolor="#0b0e17", linewidth=0.5)
            ax.bar_label(bars, fmt="%.2f", padding=4, color=MUTED, fontsize=9)

        elif chart_type == "line":
            ax.plot(labels, values, color="#f0b429", linewidth=2,
                    marker="o", markersize=4, markerfacecolor="#26a69a")
            ax.fill_between(range(len(values)), values, alpha=0.12, color="#f0b429")
            ax.set_xticks(range(len(labels)))
            ax.set_xticklabels(labels, rotation=45 if len(labels) > 8 else 0, ha="right")

        elif chart_type == "pie":
            wedges, texts, autotexts = ax.pie(
                values, labels=labels,
                colors=COLORS[:len(values)],
                autopct="%1.1f%%", startangle=90,
                textprops={"color": "#d1d4dc"},
                wedgeprops={"edgecolor": "#131722", "linewidth": 1.5},
            )
            for at in autotexts:
                at.set_color("#0b0e17")
                at.set_fontweight("bold")

        elif chart_type == "scatter":
            ax.scatter(labels, values, color="#f0b429",
                       edgecolors="#26a69a", linewidths=0.8, s=70)

        elif chart_type == "hist":
            ax.hist(values, bins="auto", color="#f0b429",
                    edgecolor="#0b0e17", linewidth=0.5, alpha=0.9)

        else:
            return f"Tipo '{chart_type}' no soportado. Disponibles: bar, line, pie, scatter, hist."

        ax.set_title(title, color="#d1d4dc", fontsize=13, pad=14)
        if x_label: ax.set_xlabel(x_label, color=MUTED)
        if y_label: ax.set_ylabel(y_label, color=MUTED)
        ax.tick_params(colors=MUTED)
        for spine in ax.spines.values():
            spine.set_edgecolor("#2a3045")
        ax.yaxis.set_major_formatter(
            mticker.FuncFormatter(lambda x, _: f"{x:,.0f}" if x >= 1000 else f"{x:.2f}")
        )

        plt.tight_layout()

        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=130, bbox_inches="tight")
        plt.close(fig)
        buf.seek(0)

        return base64.b64encode(buf.read()).decode()
    except Exception as ex:
        return f"⚠️ Unexpected Exception in tool 'plot_data' = {str(ex)}"

In [ ]:
prompt_analysis = """
    ### Eres un agente analista experto en visualización y estructuraración de datos.

    ### Cuando recibas datos:
    1. ANALIZA el contexto y el tipo de datos. \
    2. Decide el tipo de gráfica más ADECUADO (bar, line, pie, scatter, hist). \
    3. LLAMA a la herramienta 'plot_data' con los datos, el tipo elegido y títulos descriptivos. \
    4. Explica BREVEMENTE por qué elegiste ese tipo de gráfica. \
    5. Describe los patrones o insights más RELEVANTES que muestra la gráfica. \
    6. NUNCA generes gráficos por ti mismo siempre usa la herramienta 'plot_data'. \
    7. ES OBLIGATORIO usar la herramienta 'plot_data' cuando te pasen datos. \
    8. UNICAMENTE EXPLICA brevemente por qué elegiste ese tipo de gráfica y USA a la herramienta 'plot_data' con los datos, el tipo elegido y títulos descriptivos.

    ### Criterios de selección:
    - 'bar'  → comparaciones entre categorías, rankings
    - 'line' → evolución temporal, tendencias
    - 'pie'  → composición porcentual (máx. 7 categorías)
    - 'scatter' → relación entre variables
    - 'hist' → distribución de una variable continua
"""

In [ ]:
# modelo gemini -> Analista
agente_analista = Agent(
    name="Agente Analista",
    instructions=prompt_analysis,
    tools=[plot_data],
    model=gemini_model,
    handoff_description="Agente que analiza la estructura de datos y genera una grafica de los datos dinámicamente "
)

### Agente Supervisor

In [ ]:
from agents import GuardrailFunctionOutput, output_guardrail, Runner
from typing import List

In [ ]:
# Formato de Evaluacion
class ReportEvaluator(BaseModel):
    es_coherente : bool = Field(description="¿La información resultante es coherente y relevante para el reporte?")
    faltan_datos : bool = Field(description="¿Faltan datos relevantes en el reporte?")
    descripcion : str = Field(description="Descripción general del reporte generado por el agente")
    graficas_generadas : List[str] = Field(description="Lista de gráficas generadas (ej. base64 PNGs o paths temporales) para el reporte")
    observaciones : str = Field(description="Notas adicionales del evaluador sobre calidad, consistencia o mejoras")

In [ ]:
prompt_supervisor = """
    ### Eres un agente especializado en SUPERVISAR Coherencia, Falta de Datos, Gráficas Generadas para un reporte de investigación tu deber es dar OBSERVACIONES para evaluar la calidad, consistencia o mejoras del reporte. \

    ### Cuando te pasen el reporte generado:
    1. Evalúa la COHERENCIA del reporte si es válido o no es válido . \
    2. Evalúa si faltan o no faltan datos RELEVANTES en el reporte. \
    3. Evalúa las gráficas para un reporte formal y especializado, deben ser de calidad DECENTE y por lo menos debe haber UNA gráfica en el reporte. \
    4. Si FALTAN DATOS, no hay COHERENCIA o SENTIDO en el reporte, NO HAY gráficas en el reporte o la Descripcion del reporte NO ES CLARA tu deber es anotarlo en observaciones y NO GENERAR EL REPORTE FINAL. \
    5. El reporte debe contener LAS EVALUACIONES NECESARIAS para entregar un reporte consistente y formal, cumpliendo con los puntos anteriores. \
    6. Si el reporte SI CUMPLE con los puntos anteriores de igual forma anota en observaciones evaluaciones sobre calidad, consistencia o mejoras. \
    7. Si NO CUMPLE con los puntos anteriores tu deber es seguir el punto número 4. y NO GENERAR EL REPORTE FINAL, anota en las observaciones del  reporte los puntos a mejorar y el porqué falló la evaluación de calidad o consistencia. \
    8. Los Gráficos deben tener SENTIDO Y COHERENCIA según la descripción y contexto de los datos.

    ### Criterios de evaluación:
    -es_coherente ->  ¿La información resultante es coherente y relevante para el reporte?.
    -faltan_datos -> ¿Faltan datos relevantes en el reporte?.
    -descripcion -> Descripción general del reporte generado por el agente.
    -graficas_generadas -> Lista de gráficas generadas (ej. base64 PNGs o paths temporales) para el reporte
    -observaciones -> Notas adicionales sobre calidad, consistencia o mejoras.
"""

In [ ]:
# modelo groq -> supervisor
agente_supervisor = Agent(
    name="Agente Supervisor",
    instructions=prompt_supervisor,
    output_type=ReportEvaluator,
    model=groq_model
)

In [ ]:
@output_guardrail
async def report_guardrail(ctx, agent, input_data) -> GuardrailFunctionOutput:
    """ Guardrail para la evaluacion de los reportes generados """
    result = await Runner.run(agente_supervisor, input_data, context=ctx.context)
    report : ReportEvaluator = result.final_output
    is_valid = (
        report.es_coherente and
        not report.faltan_datos and
        len(report.graficas_generadas) > 0
    )
    tripwire_triggered = (report.faltan_datos or len(report.graficas_generadas) == 0 or not report.es_coherente)
    return GuardrailFunctionOutput(
        output_info={"isValid" : is_valid, "description": report.descripcion, "observaciones": report.observaciones},
        tripwire_triggered=tripwire_triggered
    )

### Agente Orquestador

In [ ]:
import  requests
from fpdf import FPDF
from PIL import Image
from io import BytesIO

In [ ]:
po_token_user = os.getenv("PUSHOVER_USER")
po_token_pass = os.getenv("PUSHOVER_TOKEN")
pushover_url = 'https://api.pushover.net/1/messages.json'

if po_token_user and po_token_pass:
    print(f'PUSHOVER_USER: {po_token_user[:8]}')
    print(f'PUSHOVER_TOKEN: {po_token_pass[:8]}')
else:
    print("PUSHOVER_USER or PUSHOVER_TOKEN are not set")

In [ ]:
@function_tool
def save_report(charts: List[str], description: str) -> str:
    """Genera un reporte final en formato PDF persistido en disco"""
    try:
        # Crear carpeta destino si no existe
        output_dir = os.path.join("..", "pdf")
        os.makedirs(output_dir, exist_ok=True)

        # Nombre del archivo
        output_path = os.path.join(output_dir, "reporte_final.pdf")

        # Inicializar PDF
        pdf = FPDF()
        pdf.set_auto_page_break(auto=True, margin=15)
        pdf.add_page()
        pdf.set_font("Arial", size=12)

        # Agregar descripción
        pdf.multi_cell(0, 10, f"Descripción del reporte:\n{description}")
        pdf.ln(10)

        # Agregar gráficas
        for idx, chart_b64 in enumerate(charts, start=1):
            try:
                # Decodificar base64 a imagen
                img_data = base64.b64decode(chart_b64)
                img = Image.open(BytesIO(img_data))

                # Guardar temporalmente
                temp_path = os.path.join(output_dir, f"temp_chart_{idx}.png")
                img.save(temp_path, "PNG")

                # Insertar en PDF
                pdf.add_page()
                pdf.cell(0, 10, f"Gráfica {idx}", ln=True)
                pdf.image(temp_path, x=10, y=30, w=180)

                # Eliminar temporal
                os.remove(temp_path)
            except Exception as e:
                pdf.add_page()
                pdf.multi_cell(0, 10, f"⚠️ Error al procesar gráfica {idx}: {str(e)}")

        # Guardar PDF
        pdf.output(output_path)

        return f"✅ Reporte guardado en {output_path}"

    except Exception as ex:
        return f"❌ Error al generar reporte: {str(ex)}"

In [ ]:
@function_tool
def push(message : str) -> None:
    """ Envía una notificación al dispositivo del usuario """
    try:
        print(f'Push Message: {message}')
        payload = {"user" : po_token_user, "token" : po_token_pass, "message" : message}
        requests.post(pushover_url, data=payload)
    except Exception as ex:
        raise f" ⚠️ Unexpected Exception in tool 'push' = {str(ex)}"

In [ ]:
# Formato del reporte final
class FinalReport(BaseModel):
    titulo_reporte : str = Field(description="Titulo del reporte")
    razonamiento : str = Field(description="Razonamiento del porque es un reporte valido")
    observaciones : str = Field(description="Observaciones del reporte")
    descripcion : str = Field(description="Descripcion del reporte")
    graficas_generadas : List[str] = Field(description="Lista de gráficas generadas (ej. base64 PNGs o paths temporales) para el reporte")

In [ ]:
prompt_orquestador = """
    ### Eres un agente especializado en ORQUESTAR y tu deber es usar el agente 'agente_investigador' para investigar de manera profunda y estructurar los datos con numeros representativos sobre el tema que pida el usuario y pasarlo al agente 'agente_analista' para que genere gráficos para el reporte, el usuario te puede preguntar sobre temas especificos y tu deber es usar el agente 'agente_investigador' para investigar y el agente 'agente_analista' en conjunto para generar un reporte. \

    ### Cuando el usuario pregunte sobre un tema:
    1. Usa el agente 'agente_investigador' que investiga y estructura los datos sobre un tema. \
    2. Si te hace FALTA CONTEXTO tu deber es preguntarle al usuario antes de usar el agente 'agente_investigador'. \
    3. NUNCA asumas nada sobre el tema SIEMPRE es necesario preguntarle al usuario si su pregunta o su tarea NO ESTA CLARA. \
    4. NUNCA investigues por tu cuenta, con el contexto necesario usa al agente 'agente_investigador' para usar los datos mas actualizados. \
    5. Una vez hecha la investigacion profunda con el agente 'agente_investigador', usa al agente 'agente_analista' para usar la informacion investigada para un analisis y generar graficas. \
    6. Es OBLIGATORIO usar a los agentes 'agente_investigador' y 'agente_analista' en conjunto para generar graficas representativas de los datos investigados con los datos mas actuales de la web. \
    7. Si la grafica generada por el agente 'agente_analista' es clara, consisa, coherente y decente tu deber es usar esa gráfica en el tool 'save_report' para persistir el reporte final generado. \
    8. USA el tool 'save_report' para guardar el reporte en formato PDF en disco y despues usa la herramienta 'push' para notificar al usuario el guardado de su reporte en pdf. \
    9. SIEMPRE identifica de que se trata el reporte, debe contener minimo UNA GRAFICA y descripciones y titulos claros, la herramienta 'save_report' de guardar un reporte decente, formal y claro sobre el tema que pidió el usuario para investigar. \
    10. Al usar la herramienta 'push' siempre debe tener un texto representativo sobre que contiene el reporte guardado para darle a entender al usuario que tipo de reporte tiene listo. \
    11. NUNCA GENERES REPORTES POR TU CUENTA, SIEMPRE usa al agente 'agente_investigador' para extraer y estructurar los datos mas actuales de la web y al agente 'agente_analista' para analisas la estructura de los datos investigados para generar la gráfica representativa del tema investigado. \
    12. SOLO si los reportes pasan el proceso de EVALUACION DE CALIDAD del output guardrail 'report_guardrail' usa la herramienta 'save_report' para generar el reporte en formato PDF y persistirlo en disco y despues usa la herramienta 'push' para notificarle al usuario que su reporte del tema que pidió está listo.

    ### Criterios para el reporte final:
    -titulo_reporte -> Titulo del reporte.
    -razonamiento -> Razonamiento del porque es un reporte valido.
    -observaciones -> Observaciones del reporte.
    -descripcion -> Descripcion del reporte.
    graficas_generadas - > Lista de gráficas generadas (ej. base64 PNGs o paths temporales) para el reporte
"""

In [ ]:
# modelo hugging-face (DeepSeek) -> orquestador
agente_orquestador = Agent(
    name="Agente Orquestador",
    instructions=prompt_orquestador,
    handoffs=[agente_investigador, agente_analista],
    output_guardrails=[report_guardrail],
    tools=[save_report, push],
    output_type=FinalReport,
    model=deepseek_model
)

### Casos de Uso

In [ ]:
from agents import OutputGuardrailTripwireTriggered, trace

In [ ]:
message = "Investiga el mercado de vehículos eléctricos en México, analiza los precios, genera una gráfica de barras y entrégame un reporte en PDF"

In [ ]:
try:
    with trace("Agencia de Consultoría IA Autónoma"):
        result = await Runner.run(agente_orquestador, message)
        report : FinalReport = result.final_output
        print(f'Agent output:\n{report.razonamiento}')

except OutputGuardrailTripwireTriggered as ex:
    print(f' ⚠️ Report Evaluator Output Guardrail triggered:\n{str(ex)}')
except Exception as ex:
    print(f" ❌ Unexpected Exception in 'agente_orquestador ' = {str(ex)}")